In [ ]:
%%spark

import os
import re
import sys
import traceback
from typing import Dict, Optional
from pyspark.sql import DataFrame, SparkSession
from datetime import date, datetime, time
from decimal import Decimal, InvalidOperation
from pyspark.sql.functions import col, md5


def publicar_tabelas_ando(
    df: DataFrame,
    database: str,
    tabela: str,
    modo: str,
    coluna_origem: str = "CD_CLI",
) -> None:

    try:
        if not database or not tabela:
            print("[ERRO] Database ou tabela não informados. Publicação não realizada.")
            return

        tbl_ando = f"{database}.ANDO_{tabela}"
        col_ando = f"DA_{coluna_origem}"

        if coluna_origem not in df.columns:
            print(
                f"[ALERTA] Coluna origem '{coluna_origem}' não existe. "
                f"Tabela '{tbl_ando}' NÃO será publicada."
            )
            return

        try:
            df_tmp = (
                df.withColumn(
                    col_ando,
                    md5(col(coluna_origem).cast("string"))
                )
                .drop(coluna_origem)
            )

            colunas_finais = [
                col_ando if c == coluna_origem else c
                for c in df.columns
            ]

            df_ando = df_tmp.select(*colunas_finais)

        except Exception as e:
            print(
                f"[ERRO] Falha ao gerar coluna '{col_ando}'. "
                f"Tabela '{tbl_ando}' NÃO publicada. Erro: {e}"
            )
            return

        try:
            (
                df_ando.write
                .mode(modo)
                .insertInto(tbl_ando)
            )

            print(
                f"[OK] Publicação concluída: '{tbl_ando}' "
                f"com '{col_ando}' (CHAR(32)) e sem exposição de "
                f"'{coluna_origem}'."
            )

        except Exception as e:
            print(f"[ERRO] Falha ao gravar tabela '{tbl_ando}': {e}")

    except Exception as e:
        print(f"[ERRO CRÍTICO] Falha geral na função: {e}")


def ler_variavel_ambiente_spark(nome_variavel: str) -> str:
    valor = os.environ.get(nome_variavel)

    if valor is None or not valor.strip():
        raise ValueError(
            f"ETL abortado: variável de ambiente '{nome_variavel}' não informada"
        )

    return valor.strip()


def registrar_erro_rotina(etapa, exc):
    logger_rotina.error(
        f"[ROTINA][ERRO] Falha na etapa {etapa}. "
        f"tipo={type(exc).__name__} mensagem={str(exc)}"
    )
    logger_rotina.error(traceback.format_exc())


def _use_logs(default=True) -> bool:
    value = os.environ.get("USE_LOGS")

    if value is None:
        return default

    return str(value).strip().lower() == "true"


def _flush() -> None:
    try:
        sys.stdout.flush()
    except Exception:
        pass

    try:
        sys.stderr.flush()
    except Exception:
        pass


def _print_log(msg: str) -> None:
    print(msg)
    _flush()


class ScreenLogger:
    def __init__(self, name: str = "PIPELINE") -> None:
        self.name = name

    def step(self, suffix: str) -> "ScreenLogger":
        return ScreenLogger(name=f"{self.name}.{suffix}")

    def _prefix(self) -> str:
        return f"[{self.name}] "

    def info(self, msg: str) -> None:
        if not _use_logs(True):
            return

        _print_log(f"{self._prefix()}{msg}")

    def error(self, msg: str) -> None:
        if not _use_logs(True):
            return

        _print_log(f"{self._prefix()}{msg}")

    def obj(self, value, title: str = None) -> None:
        if not _use_logs(True):
            return

        if title:
            _print_log(f"{self._prefix()}{title}")

        _print_log(str(value))

    def df(self, df, n: int = 20, truncate=True, title: str = None) -> None:
        if not _use_logs(True):
            return

        if title:
            _print_log(f"{self._prefix()}{title}")

        df.show(n=n, truncate=truncate)
        _flush()

    def df_schema(self, df, title: str = None) -> None:
        if not _use_logs(True):
            return

        if title:
            _print_log(f"{self._prefix()}{title}")

        df.printSchema()
        _flush()


class NullLogger:
    def step(self, suffix: str):
        return self

    def info(self, msg: str) -> None:
        pass

    def error(self, msg: str) -> None:
        pass

    def obj(self, value, title: str = None) -> None:
        pass

    def df(self, df, n: int = 20, truncate=True, title: str = None) -> None:
        pass

    def df_schema(self, df, title: str = None) -> None:
        pass


def criar_logger_spark(nome: str = "PIPELINE"):
    if _use_logs(True):
        return ScreenLogger(nome)

    return NullLogger()


logger = criar_logger_spark("PIPELINE")
logger_rotina = logger


class ClienteOracleSpark:
    DEFAULT_DRIVER = "oracle.jdbc.OracleDriver"

    def __init__(
        self,
        spark: SparkSession,
        env: Optional[Dict[str, str]] = None,
    ) -> None:
        self.spark = spark
        self.env = env or dict(os.environ)

        def env_required(key: str) -> str:
            value = self.env.get(key)

            if value is None or str(value).strip() == "":
                raise ValueError(f"Variável obrigatória não encontrada: {key}")

            return str(value).strip()

        def env_optional(key: str) -> Optional[str]:
            value = self.env.get(key)

            if value is None or str(value).strip() == "":
                return None

            return str(value).strip()

        self.user = env_required("VDP_ORACLE_USER")
        self.password = env_required("VDP_ORACLE_PASSWORD")
        self.host_1 = env_required("VDP_ORACLE_HOST_1")
        self.host_2 = env_optional("VDP_ORACLE_HOST_2")
        self.port = env_required("VDP_ORACLE_PORTA")
        self.service_name = env_optional("VDP_ORACLE_SERVICE_NAME") or env_required(
            "VDP_ORACLE_SERVICE"
        )
        self.schema = (env_optional("VDP_ORACLE_SCHEMA") or self.user).upper()
        self.driver = env_optional("VDP_ORACLE_DRIVER") or self.DEFAULT_DRIVER
        self.jar_path = env_optional("VDP_ORACLE_JAR") or "/dados/shared/bin/ojdbc8.jar"

        if self.driver != self.DEFAULT_DRIVER:
            raise ValueError(f"Driver Oracle inválido: {self.driver}")

        if self.host_2:
            self.url = (
                "jdbc:oracle:thin:@(DESCRIPTION="
                "(LOAD_BALANCE=OFF)"
                "(FAILOVER=ON)"
                "(CONNECT_TIMEOUT=10)"
                "(TRANSPORT_CONNECT_TIMEOUT=3)"
                "(RETRY_COUNT=3)"
                "(ADDRESS_LIST="
                f"(ADDRESS=(PROTOCOL=TCP)(HOST={self.host_1})(PORT={self.port}))"
                f"(ADDRESS=(PROTOCOL=TCP)(HOST={self.host_2})(PORT={self.port}))"
                ")"
                f"(CONNECT_DATA=(SERVICE_NAME={self.service_name}))"
                ")"
            )
        else:
            self.url = f"jdbc:oracle:thin:@//{self.host_1}:{self.port}/{self.service_name}"

    def run_select(
        self,
        sql: str,
        fetchsize: Optional[int] = None,
    ) -> DataFrame:
        query = (sql or "").strip()

        if not query:
            raise ValueError("sql nao pode ser vazio")

        if query.endswith(";"):
            query = query[:-1].strip()

        reader = (
            self.spark.read
            .format("jdbc")
            .option("url", self.url)
            .option("driver", self.driver)
            .option("user", self.user)
            .option("password", self.password)
            .option("dbtable", f"({query}) T")
        )

        if fetchsize is not None:
            if int(fetchsize) <= 0:
                raise ValueError("fetchsize deve ser maior que zero")

            reader = reader.option("fetchsize", int(fetchsize))

        return reader.load()

    def selecionar_tabela(
        self,
        nome_tabela: str,
        owner: Optional[str] = None,
        fetchsize: Optional[int] = None,
        show: bool = False,
        truncate: bool = True,
        n: int = 20,
    ) -> DataFrame:
        owner_final = (owner or self.schema or "").strip().upper()
        tabela_final = (nome_tabela or "").strip().upper()

        if not owner_final:
            raise ValueError("owner/schema nao pode ser vazio")

        if not tabela_final:
            raise ValueError("nome_tabela nao pode ser vazio")

        if "." in tabela_final:
            raise ValueError(
                "nome_tabela deve receber apenas o nome da tabela. "
                "Informe owner/schema separadamente."
            )

        if int(n) <= 0:
            raise ValueError("n deve ser maior que zero")

        df = self.run_select(
            sql=f"SELECT * FROM {owner_final}.{tabela_final}",
            fetchsize=fetchsize,
        )

        if show:
            df.show(n=int(n), truncate=truncate)

        return df

    def execute(self, sql: str) -> None:
        command = (sql or "").strip()

        if not command:
            raise ValueError("sql nao pode ser vazio")

        if command.endswith(";"):
            command = command[:-1].strip()

        jvm = self.spark._jvm
        conn = None
        stmt = None

        def criar_conexao_driver_manager():
            try:
                jvm.java.lang.Class.forName(self.driver)
            except Exception:
                context_loader = (
                    jvm.java.lang.Thread
                    .currentThread()
                    .getContextClassLoader()
                )

                jvm.java.lang.Class.forName(
                    self.driver,
                    True,
                    context_loader,
                )

            return jvm.java.sql.DriverManager.getConnection(
                self.url,
                self.user,
                self.password,
            )

        def criar_conexao_url_classloader():
            gateway = self.spark.sparkContext._gateway

            jar_file = jvm.java.io.File(self.jar_path)

            if not jar_file.exists():
                raise ValueError(
                    f"Jar Oracle nao encontrado em: {self.jar_path}. "
                    "Informe VDP_ORACLE_JAR ou carregue o ojdbc8.jar na sessao Spark."
                )

            jar_url = jar_file.toURI().toURL()

            urls = gateway.new_array(jvm.java.net.URL, 1)
            urls[0] = jar_url

            parent_loader = (
                jvm.java.lang.Thread
                .currentThread()
                .getContextClassLoader()
            )

            loader = jvm.java.net.URLClassLoader(urls, parent_loader)

            driver_class = jvm.java.lang.Class.forName(
                self.driver,
                True,
                loader,
            )

            driver = driver_class.newInstance()

            props = jvm.java.util.Properties()
            props.setProperty("user", self.user)
            props.setProperty("password", self.password)

            return driver.connect(self.url, props)

        try:
            try:
                conn = criar_conexao_driver_manager()
            except Exception:
                conn = criar_conexao_url_classloader()

            conn.setAutoCommit(False)

            stmt = conn.createStatement()
            stmt.execute(command)

            conn.commit()

        except Exception:
            if conn is not None:
                try:
                    conn.rollback()
                except Exception:
                    pass

            raise

        finally:
            if stmt is not None:
                try:
                    stmt.close()
                except Exception:
                    pass

            if conn is not None:
                try:
                    conn.close()
                except Exception:
                    pass

        return None

    def carregar_df(
        self,
        df: DataFrame,
        table_name: str,
        owner: Optional[str] = None,
        batchsize: int = 5000,
        num_partitions: int = 1,
    ) -> None:
        if df is None:
            raise ValueError("df nao pode ser None")

        if int(batchsize) <= 0:
            raise ValueError("batchsize deve ser maior que zero")

        if int(num_partitions) <= 0:
            raise ValueError("num_partitions deve ser maior que zero")

        table = (table_name or "").strip()

        if not table:
            raise ValueError("table_name nao pode ser vazio")

        if "." in table:
            raise ValueError(
                "table_name deve receber apenas o nome da tabela. "
                "Informe owner/schema separadamente."
            )

        final_owner = (owner or self.schema or "").strip()

        if not final_owner:
            raise ValueError("owner/schema nao pode ser vazio")

        full_table_name = f"{final_owner.upper()}.{table.upper()}"

        writer_df = df.coalesce(int(num_partitions))

        (
            writer_df.write
            .format("jdbc")
            .mode("append")
            .option("url", self.url)
            .option("driver", self.driver)
            .option("user", self.user)
            .option("password", self.password)
            .option("dbtable", full_table_name)
            .option("batchsize", int(batchsize))
            .save()
        )

        return None

    def limpar_tabela(
        self,
        table_name: str,
        owner: Optional[str] = None,
        use_truncate: bool = False,
        batchsize: int = 5000,
    ) -> None:
        if int(batchsize) <= 0:
            raise ValueError("batchsize deve ser maior que zero")

        table = (table_name or "").strip()

        if not table:
            raise ValueError("table_name nao pode ser vazio")

        if "." in table:
            raise ValueError(
                "table_name deve receber apenas o nome da tabela. "
                "Informe owner/schema separadamente."
            )

        final_owner = (owner or self.schema or "").strip()

        if not final_owner:
            raise ValueError("owner/schema nao pode ser vazio")

        full_table_name = f"{final_owner.upper()}.{table.upper()}"

        if use_truncate:
            self.execute(f"TRUNCATE TABLE {full_table_name}")
            return None

        lote = int(batchsize)

        while True:
            df_count = self.run_select(
                f"SELECT COUNT(1) AS QTD FROM {full_table_name}"
            )

            qtd_restante = int(df_count.collect()[0]["QTD"])

            if qtd_restante == 0:
                break

            self.execute(
                f"DELETE FROM {full_table_name} WHERE ROWNUM <= {lote}"
            )

        return None

    def reload_dataframe(
        self,
        df: DataFrame,
        table_name: str,
        owner: Optional[str] = None,
        batchsize: int = 5000,
        num_partitions: int = 1,
        use_truncate: bool = False,
    ) -> None:
        if df is None:
            raise ValueError("df nao pode ser None")

        table = (table_name or "").strip()

        if not table:
            raise ValueError("table_name nao pode ser vazio")

        if "." in table:
            raise ValueError(
                "table_name deve receber apenas o nome da tabela. "
                "Informe owner/schema separadamente."
            )

        final_owner = (owner or self.schema or "").strip()

        if not final_owner:
            raise ValueError("owner/schema nao pode ser vazio")

        self.limpar_tabela(
            table_name=table,
            owner=final_owner,
            use_truncate=use_truncate,
            batchsize=batchsize,
        )

        self.carregar_df(
            df=df,
            table_name=table,
            owner=final_owner,
            batchsize=batchsize,
            num_partitions=num_partitions,
        )

        return None


def criar_cliente_oracle_spark(
    env: Optional[Dict[str, str]] = None,
) -> ClienteOracleSpark:
    return ClienteOracleSpark(spark=spark, env=env)



class ClienteDb2Spark:
    DEFAULT_DRIVER = "com.ibm.db2.jcc.DB2Driver"

    def __init__(
        self,
        spark: SparkSession,
        env: Optional[Dict[str, str]] = None,
    ) -> None:
        self.spark = spark
        self.env = env or dict(os.environ)

        def env_required(key: str) -> str:
            value = self.env.get(key)

            if value is None or str(value).strip() == "":
                raise ValueError(f"Variavel obrigatoria nao encontrada: {key}")

            return str(value).strip()

        def env_optional(key: str) -> Optional[str]:
            value = self.env.get(key)

            if value is None or str(value).strip() == "":
                return None

            return str(value).strip()

        self.user = env_required("DB2_USER")
        self.password = env_required("DB2_PASSWORD")
        self.host = env_required("DB2_HOST")
        self.port = env_required("DB2_PORTA")
        self.database = env_required("DB2_DATABASE")
        self.driver = env_optional("DB2_DRIVER") or self.DEFAULT_DRIVER

        if self.driver != self.DEFAULT_DRIVER:
            raise ValueError(f"Driver DB2 invalido: {self.driver}")

        self.url = f"jdbc:db2://{self.host}:{self.port}/{self.database}"

    @staticmethod
    def _normalizar_identificador(
        valor: str,
        nome_parametro: str,
    ) -> str:
        resultado = (valor or "").strip().upper()

        if not resultado:
            raise ValueError(f"{nome_parametro} nao pode ser vazio")

        if not re.fullmatch(
            r"[A-Z][A-Z0-9_#$@]{0,127}",
            resultado,
        ):
            raise ValueError(
                f"{nome_parametro} possui formato invalido: {valor!r}. "
                "Use identificador DB2 simples, sem aspas e sem ponto."
            )

        return resultado

    @staticmethod
    def _normalizar_bound(valor) -> str:
        if isinstance(valor, datetime):
            return valor.strftime("%Y-%m-%d %H:%M:%S")

        if isinstance(valor, date):
            return valor.isoformat()

        texto = str(valor).strip()

        if not texto:
            raise ValueError(
                "lower_bound e upper_bound nao podem ser vazios"
            )

        return texto

    @staticmethod
    def _converter_bound_comparavel(valor: str):
        try:
            return Decimal(valor)
        except InvalidOperation:
            pass

        try:
            if len(valor) == 10:
                return datetime.combine(
                    date.fromisoformat(valor),
                    time.min,
                )

            return datetime.fromisoformat(valor)

        except ValueError as exc:
            raise ValueError(
                "Bounds devem ser numericos ou data/timestamp ISO. "
                "Exemplos: 1, 100000, '2026-06-01', "
                "'2026-06-01 00:00:00'."
            ) from exc

    @staticmethod
    def _normalizar_data_filtro(valor) -> str:
        if isinstance(valor, datetime):
            return valor.strftime("%Y-%m-%d %H:%M:%S")

        if isinstance(valor, date):
            return valor.isoformat()

        texto = str(valor).strip()

        if not texto:
            raise ValueError("filtrar_data nao pode conter valor vazio")

        return texto

    @staticmethod
    def _literal_sql(valor: str) -> str:
        return "'" + valor.replace("'", "''") + "'"

    @staticmethod
    def _formatar_lista_copiavel(valores: list) -> str:
        if not valores:
            return "[]"

        linhas = ["["]
        linhas.extend(f'    "{valor}",' for valor in valores)
        linhas.append("]")

        return "\n".join(linhas)

    @staticmethod
    def _estimar_tamanho_coluna(tipo: str, tamanho) -> int:
        try:
            tamanho_final = max(0, int(tamanho or 0))
        except (TypeError, ValueError):
            tamanho_final = 0

        tamanhos_fixos = {
            "SMALLINT": 2,
            "INTEGER": 4,
            "BIGINT": 8,
            "REAL": 4,
            "FLOAT": 8,
            "DOUBLE": 8,
            "DATE": 4,
            "TIME": 3,
            "TIMESTAMP": 12,
            "TIMESTMP": 12,
        }

        if tipo in tamanhos_fixos:
            return tamanhos_fixos[tipo]

        if tipo in {"GRAPHIC", "VARGRAPHIC", "LONGVARG"}:
            return tamanho_final * 2

        return tamanho_final or 32

    def _obter_config_int(self, chave: str) -> Optional[int]:
        try:
            valor = self.spark.sparkContext.getConf().get(chave)
            return int(valor) if valor is not None else None
        except (TypeError, ValueError, AttributeError):
            return None

    def run_select(
        self,
        sql: str,
        fetchsize: Optional[int] = None,
        partition_column: Optional[str] = None,
        lower_bound=None,
        upper_bound=None,
        num_partitions: Optional[int] = None,
        query_timeout: Optional[int] = None,
        show: bool = False,
        truncate: bool = True,
        n: int = 20,
    ) -> DataFrame:
        query = (sql or "").strip()

        if not query:
            raise ValueError("sql nao pode ser vazio")

        if query.endswith(";"):
            query = query[:-1].strip()

        if int(n) <= 0:
            raise ValueError("n deve ser maior que zero")

        reader = (
            self.spark.read
            .format("jdbc")
            .option("url", self.url)
            .option("driver", self.driver)
            .option("user", self.user)
            .option("password", self.password)
            .option("dbtable", f"({query}) T")
        )

        if fetchsize is not None:
            fetchsize_final = int(fetchsize)

            if fetchsize_final <= 0:
                raise ValueError("fetchsize deve ser maior que zero")

            reader = reader.option("fetchsize", fetchsize_final)

        if query_timeout is not None:
            query_timeout_final = int(query_timeout)

            if query_timeout_final <= 0:
                raise ValueError("query_timeout deve ser maior que zero")

            reader = reader.option("queryTimeout", query_timeout_final)

        parametros_particao = [
            partition_column,
            lower_bound,
            upper_bound,
            num_partitions,
        ]

        if any(valor is not None for valor in parametros_particao):
            if any(valor is None for valor in parametros_particao):
                raise ValueError(
                    "Para leitura particionada, informe partition_column, "
                    "lower_bound, upper_bound e num_partitions."
                )

            partition_column_final = self._normalizar_identificador(
                partition_column,
                "partition_column",
            )

            lower_bound_final = self._normalizar_bound(lower_bound)
            upper_bound_final = self._normalizar_bound(upper_bound)
            num_partitions_final = int(num_partitions)

            if (
                self._converter_bound_comparavel(lower_bound_final)
                >= self._converter_bound_comparavel(upper_bound_final)
            ):
                raise ValueError(
                    "lower_bound deve ser menor que upper_bound"
                )

            if num_partitions_final <= 0:
                raise ValueError(
                    "num_partitions deve ser maior que zero"
                )

            reader = (
                reader
                .option("partitionColumn", partition_column_final)
                .option("lowerBound", lower_bound_final)
                .option("upperBound", upper_bound_final)
                .option("numPartitions", num_partitions_final)
            )

        df = reader.load()

        if show:
            df.show(n=int(n), truncate=truncate)

        return df

    def selecionar_tabela(
        self,
        schema: str,
        nome_tabela: str,
        filtrar_cols: Optional[list] = None,
        col_data: Optional[str] = None,
        filtrar_data=None,
        col_particao: Optional[str] = None,
        lower_bound=None,
        upper_bound=None,
        fetchsize: Optional[int] = None,
        num_partitions: Optional[int] = None,
        show: bool = False,
        truncate: bool = True,
        n: int = 20,
    ) -> DataFrame:
        """
        Le regras:
        - Sem argumentos opcionais, le a tabela inteira.
        - filtrar_cols=None ou [] gera SELECT *.
        - Uma data gera COL_DATA >= data.
        - Duas datas geram COL_DATA >= inicio AND COL_DATA < fim.
        - Nunca usa <= no filtro de data.
        """
        schema_final = self._normalizar_identificador(schema, "schema")
        tabela_final = self._normalizar_identificador(
            nome_tabela,
            "nome_tabela",
        )

        if int(n) <= 0:
            raise ValueError("n deve ser maior que zero")

        col_data_final = (
            self._normalizar_identificador(col_data, "col_data")
            if col_data is not None
            else None
        )

        col_particao_final = (
            self._normalizar_identificador(
                col_particao,
                "col_particao",
            )
            if col_particao is not None
            else None
        )

        colunas_retorno = []

        if filtrar_cols:
            if isinstance(filtrar_cols, str):
                raise TypeError(
                    "filtrar_cols deve ser uma lista de nomes de colunas"
                )

            for coluna in filtrar_cols:
                coluna_final = self._normalizar_identificador(
                    coluna,
                    "filtrar_cols",
                )

                if coluna_final not in colunas_retorno:
                    colunas_retorno.append(coluna_final)

        colunas_leitura = list(colunas_retorno)
        remover_coluna_particao_apos_leitura = False

        if (
            colunas_leitura
            and col_particao_final
            and col_particao_final not in colunas_leitura
        ):
            colunas_leitura.append(col_particao_final)
            remover_coluna_particao_apos_leitura = True

        selecao_sql = (
            ",\n    ".join(colunas_leitura)
            if colunas_leitura
            else "*"
        )

        valores_data = []

        if filtrar_data is not None:
            if isinstance(filtrar_data, (str, date, datetime)):
                valores_data = [filtrar_data]

            elif isinstance(filtrar_data, (list, tuple)):
                valores_data = [
                    valor
                    for valor in filtrar_data
                    if valor is not None
                    and str(valor).strip() != ""
                ]

            else:
                raise TypeError(
                    "filtrar_data deve receber uma data ou uma lista/tupla "
                    "com uma ou duas datas."
                )

            if len(valores_data) > 2:
                raise ValueError(
                    "filtrar_data aceita somente uma data ou duas datas: "
                    "(inicio, fim)."
                )

            if valores_data and not col_data_final:
                raise ValueError(
                    "Informe col_data quando utilizar filtrar_data."
                )

        where_sql = ""

        if len(valores_data) == 1:
            data_inicial = self._normalizar_data_filtro(valores_data[0])

            where_sql = (
                f"\nWHERE {col_data_final} >= "
                f"{self._literal_sql(data_inicial)}"
            )

        elif len(valores_data) == 2:
            data_inicial = self._normalizar_data_filtro(valores_data[0])
            data_final = self._normalizar_data_filtro(valores_data[1])

            if (
                self._converter_bound_comparavel(data_inicial)
                >= self._converter_bound_comparavel(data_final)
            ):
                raise ValueError(
                    "No filtro de data, o inicio deve ser menor que o fim."
                )

            where_sql = (
                f"\nWHERE {col_data_final} >= "
                f"{self._literal_sql(data_inicial)}"
                f"\n  AND {col_data_final} <= "
                f"{self._literal_sql(data_final)}"
            )

        sql = (
            "SELECT\n"
            f"    {selecao_sql}\n"
            f"FROM {schema_final}.{tabela_final}"
            f"{where_sql}"
        )

        df = self.run_select(
            sql=sql,
            fetchsize=fetchsize,
            partition_column=col_particao_final,
            lower_bound=lower_bound,
            upper_bound=upper_bound,
            num_partitions=num_partitions,
            show=False,
        )

        if remover_coluna_particao_apos_leitura:
            df = df.drop(col_particao_final)

        if show:
            df.show(n=int(n), truncate=truncate)

        return df

    def diagnosticar_tabela(
        self,
        schema: str,
        nome_tabela: str,
        show: bool = True,
        truncate: bool = False,
    ) -> dict:
        """
        Retorna somente os insumos para preencher selecionar_tabela(...):
        filtrar_cols, col_data, col_particao, bounds, fetchsize
        e num_partitions.
        """
        schema_final = self._normalizar_identificador(schema, "schema")
        tabela_final = self._normalizar_identificador(
            nome_tabela,
            "nome_tabela",
        )

        sql_tabela = f"""
            SELECT
                TYPE AS TIPO_OBJETO,
                CARDF AS ESTIMATIVA_LINHAS
            FROM SYSIBM.SYSTABLES
            WHERE CREATOR = '{schema_final}'
              AND NAME = '{tabela_final}'
        """

        metadados_tabela = self.run_select(sql_tabela).collect()

        if not metadados_tabela:
            raise ValueError(
                f"Tabela ou view nao encontrada: "
                f"{schema_final}.{tabela_final}"
            )

        try:
            estimativa_linhas = float(
                metadados_tabela[0]["ESTIMATIVA_LINHAS"]
            )
        except (TypeError, ValueError):
            estimativa_linhas = -1

        sql_colunas = f"""
            SELECT
                NAME AS COLUNA,
                COLNO AS POSICAO,
                COLTYPE AS TIPO_DB2,
                LENGTH AS TAMANHO,
                NULLS AS ACEITA_NULO
            FROM SYSIBM.SYSCOLUMNS
            WHERE TBCREATOR = '{schema_final}'
              AND TBNAME = '{tabela_final}'
            ORDER BY COLNO
        """

        colunas = self.run_select(sql_colunas).collect()

        if not colunas:
            raise ValueError(
                f"Nenhuma coluna encontrada para "
                f"{schema_final}.{tabela_final}"
            )

        sql_indices = f"""
            SELECT
                K.COLNAME AS COLUNA,
                K.COLSEQ AS POSICAO_INDICE
            FROM SYSIBM.SYSINDEXES I
            INNER JOIN SYSIBM.SYSKEYS K
                ON I.CREATOR = K.IXCREATOR
               AND I.NAME = K.IXNAME
            WHERE I.TBCREATOR = '{schema_final}'
              AND I.TBNAME = '{tabela_final}'
            ORDER BY
                K.COLNAME,
                K.COLSEQ
        """

        indices = self.run_select(sql_indices).collect()

        colunas_indice_lider = set()

        for indice in indices:
            try:
                posicao_indice = int(indice["POSICAO_INDICE"])
            except (TypeError, ValueError):
                continue

            if posicao_indice == 1:
                colunas_indice_lider.add(
                    str(indice["COLUNA"]).strip().upper()
                )

        tipos_data = {
            "DATE",
            "TIMESTAMP",
            "TIMESTMP",
        }

        tipos_numericos = {
            "SMALLINT",
            "INTEGER",
            "BIGINT",
            "DECIMAL",
            "NUMERIC",
            "DECFLOAT",
            "REAL",
            "FLOAT",
            "DOUBLE",
        }

        tipos_lob = {
            "BLOB",
            "CLOB",
            "DBCLOB",
            "XML",
            "LONGVAR",
            "LONGVARG",
            "LONGVARB",
        }

        tipos_particao = tipos_data | tipos_numericos

        filtrar_cols = []
        col_data = []
        col_particao = []
        tipo_por_coluna = {}

        tamanho_linha_estimado = 0
        possui_lob = False

        for coluna in colunas:
            nome_coluna = str(coluna["COLUNA"]).strip().upper()
            tipo_coluna = str(coluna["TIPO_DB2"] or "").strip().upper()
            aceita_nulo = str(
                coluna["ACEITA_NULO"] or ""
            ).strip().upper()

            filtrar_cols.append(nome_coluna)
            tipo_por_coluna[nome_coluna] = tipo_coluna

            tamanho_linha_estimado += self._estimar_tamanho_coluna(
                tipo=tipo_coluna,
                tamanho=coluna["TAMANHO"],
            )

            if tipo_coluna in tipos_lob:
                possui_lob = True

            if tipo_coluna in tipos_data:
                col_data.append(nome_coluna)

            if (
                tipo_coluna in tipos_particao
                and aceita_nulo == "N"
                and nome_coluna in colunas_indice_lider
            ):
                col_particao.append(nome_coluna)

        bounds = {}

        if col_particao:
            agregacoes = []

            for posicao, nome_coluna in enumerate(col_particao):
                agregacoes.extend([
                    f"MIN({nome_coluna}) AS LB_{posicao}",
                    f"MAX({nome_coluna}) AS UB_{posicao}",
                ])

            agregacoes_sql = ",\n                    ".join(agregacoes)

            sql_bounds = f"""
                SELECT
                    {agregacoes_sql}
                FROM {schema_final}.{tabela_final}
            """

            linha_bounds = (
                self.run_select(sql_bounds)
                .collect()[0]
                .asDict(recursive=True)
            )

            for posicao, nome_coluna in enumerate(col_particao):
                lower_bound_valor = linha_bounds.get(f"LB_{posicao}")
                upper_bound_valor = linha_bounds.get(f"UB_{posicao}")

                bounds[nome_coluna] = {
                    "lower_bound": (
                        self._normalizar_bound(lower_bound_valor)
                        if lower_bound_valor is not None
                        else None
                    ),
                    "upper_bound": (
                        self._normalizar_bound(upper_bound_valor)
                        if upper_bound_valor is not None
                        else None
                    ),
                }

        if possui_lob or tamanho_linha_estimado > 16_384:
            fetchsize_sugerido = 1_000
        elif tamanho_linha_estimado > 4_096:
            fetchsize_sugerido = 5_000
        else:
            fetchsize_sugerido = 10_000

        try:
            capacidade_spark = max(
                1,
                int(self.spark.sparkContext.defaultParallelism),
            )
        except (TypeError, ValueError, AttributeError):
            capacidade_spark = 1

        executor_cores = self._obter_config_int("spark.executor.cores")
        executor_instances = self._obter_config_int(
            "spark.executor.instances"
        )

        if executor_cores and executor_instances:
            capacidade_spark = max(
                capacidade_spark,
                executor_cores * executor_instances,
            )

        capacidade_jdbc = min(capacidade_spark, 16)

        if not col_particao:
            num_partitions_sugerido = None
        elif estimativa_linhas == 0:
            num_partitions_sugerido = 1
        elif 0 < estimativa_linhas <= 250_000:
            num_partitions_sugerido = 1
        elif 0 < estimativa_linhas <= 2_000_000:
            num_partitions_sugerido = min(4, capacidade_jdbc)
        elif 0 < estimativa_linhas <= 20_000_000:
            num_partitions_sugerido = min(8, capacidade_jdbc)
        elif estimativa_linhas > 20_000_000:
            num_partitions_sugerido = capacidade_jdbc
        else:
            num_partitions_sugerido = min(8, capacidade_jdbc)

        if show:
            print("=" * 72)
            print("DIAGNOSTICO DB2 PARA selecionar_tabela")
            print(f"Tabela: {schema_final}.{tabela_final}")
            print("=" * 72)

            print("\nfiltrar_cols =")
            print(self._formatar_lista_copiavel(filtrar_cols))

            print("\ncol_data =")
            print(self._formatar_lista_copiavel(col_data))

            print("\ncol_particao =")
            print(self._formatar_lista_copiavel(col_particao))

            print("\nlower_bound / upper_bound por col_particao:")

            if bounds:
                linhas_bounds = [
                    (
                        nome_coluna,
                        tipo_por_coluna[nome_coluna],
                        bounds[nome_coluna]["lower_bound"],
                        bounds[nome_coluna]["upper_bound"],
                    )
                    for nome_coluna in col_particao
                ]

                df_bounds = self.spark.createDataFrame(
                    linhas_bounds,
                    [
                        "COL_PARTICAO",
                        "TIPO_DB2",
                        "LOWER_BOUND",
                        "UPPER_BOUND",
                    ],
                )

                df_bounds.show(
                    n=len(linhas_bounds),
                    truncate=truncate,
                )
            else:
                print("[]")

            print(f"\nfetchsize = {fetchsize_sugerido}")

            if num_partitions_sugerido is None:
                print(
                    "num_partitions = None "
                    "(nenhuma col_particao elegivel)"
                )
            else:
                print(
                    f"num_partitions = {num_partitions_sugerido}"
                )

            print("=" * 72)

        return {
            "filtrar_cols": filtrar_cols,
            "col_data": col_data,
            "col_particao": col_particao,
            "bounds": bounds,
            "fetchsize": fetchsize_sugerido,
            "num_partitions": num_partitions_sugerido,
        }


def criar_cliente_db2_spark(
    env: Optional[Dict[str, str]] = None,
) -> ClienteDb2Spark:
    return ClienteDb2Spark(spark=spark, env=env)

def help_gerenciador_sessao_spark_remoto() -> None:
    print("""
============================================================
gerenciador_sessao_spark_remoto.ipynb
============================================================

Este notebook deve ser executado depois da criacao da sessao Spark.

Ele disponibiliza na sessao Spark:

- ler_variavel_ambiente_spark
- criar_logger_spark
- logger
- logger_rotina
- registrar_erro_rotina
- publicar_tabelas_ando
- ClienteOracleSpark
- criar_cliente_oracle_spark
- ClienteDb2Spark
- criar_cliente_db2_spark

Clientes:

cliente_oracle = criar_cliente_oracle_spark()
cliente_db2 = criar_cliente_db2_spark()

Oracle:

cliente_oracle.run_select("SELECT * FROM OWNER.TABELA")
cliente_oracle.selecionar_tabela("TABELA", owner="OWNER")
cliente_oracle.execute("TRUNCATE TABLE OWNER.TABELA")
cliente_oracle.limpar_tabela("TABELA", owner="OWNER")
cliente_oracle.carregar_df(df, "TABELA", owner="OWNER")
cliente_oracle.reload_dataframe(df, "TABELA", owner="OWNER")

DB2:

cliente_db2.run_select("SELECT * FROM SCHEMA.TABELA")
cliente_db2.selecionar_tabela("SCHEMA", "TABELA")

============================================================
""")